# Imports

In [23]:
import pandas
import seaborn
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

# File Reading

In [24]:
awards_players = pandas.read_csv('dataset/awards_players.csv')
coaches = pandas.read_csv('dataset/coaches.csv')
players_teams = pandas.read_csv('dataset/players_teams.csv') 
players = pandas.read_csv('dataset/players.csv')
series_post = pandas.read_csv('dataset/series_post.csv')
teams_post = pandas.read_csv('dataset/teams_post.csv')
teams = pandas.read_csv('dataset/teams.csv')

# Data Cleaning

In [25]:
# lgID is not needed as its all from WNBA
awards_players.drop('lgID', axis=1, inplace=True, errors='ignore')
coaches.drop('lgID', axis=1, inplace=True, errors='ignore')
teams.drop('lgID', axis=1, inplace=True, errors='ignore')

# player firstseason and lastseason is constant (0)
players.drop(['firstseason', 'lastseason'], axis=1, inplace=True, errors='ignore')

# teams "tmORB" ,"tmDRB" ,"tmTRB" ,"opptmORB" ,"opptmDRB" ,"opptmTRB" is all 0
teams.drop(['tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB'], axis=1, inplace=True, errors='ignore')

# name and arena are constant strings, divID is empty
teams.drop(['name', 'arena', 'divID', 'confID', 'tmID'], axis=1, inplace=True, errors='ignore')

# Features

In [ ]:
teams['round_reached'] = 0
teams.loc[teams["firstRound"] == "", 'round_reached'] = 0
teams.loc[teams["firstRound"] == "L", 'round_reached'] = 1
teams.loc[teams["semis"] == "L", 'round_reached'] = 2
teams.loc[teams["finals"] == "L", 'round_reached'] = 3
teams.loc[teams["finals"] == "W", 'round_reached'] = 4
teams.drop(['firstRound', 'semis', 'finals'], axis=1, inplace=True, errors='ignore')

teams['awards_won'] = 0
for index, row in teams.iterrows():
    teamID = row['franchID']
    year = row['year'] + 1
    players_teams_year = players_teams[(players_teams['year'] == year) & (players_teams['tmID'] == teamID)]
    coaches_teams_year = coaches[(coaches['year'] == year) & (coaches['tmID'] == teamID)]
    awards_won = awards_players[(awards_players['playerID'].isin(players_teams_year['playerID'])) | (awards_players['playerID'].isin(coaches_teams_year['coachID']))]
    awards_won = awards_won[awards_won['year'] <= year]
    teams.at[index, 'awards_won'] = len(awards_won)

players.rename(columns={'bioID': 'playerID'}, inplace=True)
players_teams = players_teams.merge(players[['playerID', 'height', 'weight']], on='playerID', how='left')

average_stats = players_teams.groupby(['tmID', 'year']).agg(
    average_height=('height', 'mean'),
    average_weight=('weight', 'mean')
).reset_index()

teams['average_height'] = 0.0
teams['average_weight'] = 0.0

for index, row in teams.iterrows():
    teamID = row['franchID']
    year = row['year'] + 1
    avg_height = average_stats[(average_stats['tmID'] == teamID) & (average_stats['year'] == year)]['average_height'].values
    if len(avg_height) > 0:
        teams.at[index, 'average_height'] = avg_height[0]
    avg_weight = average_stats[(average_stats['tmID'] == teamID) & (average_stats['year'] == year)]['average_weight'].values
    if len(avg_weight) > 0:
        teams.at[index, 'average_weight'] = avg_weight[0]

teams_table = teams.copy()
teams_table.sort_values(by=['franchID', 'year'], inplace=True)
teams_table['next_year_playoff'] = teams_table['playoff'].shift(-1)
teams_table = teams_table.groupby('franchID').apply(lambda x: x.iloc[:-1])
teams_table.drop('playoff', axis=1, inplace=True)
teams_table['next_year_playoff'] = teams_table['next_year_playoff'].apply(lambda x: 1 if x == 'Y' else 0)

# Analysis

## Coaches winratio

In [ ]:
coaches_names = coaches['coachID'].unique()
coaches_winratio = coaches.groupby('coachID').agg({'won': 'sum', 'lost': 'sum'})
coaches_winratio['winratio'] = coaches_winratio['won'] / (coaches_winratio['won'] + coaches_winratio['lost'])
coaches_winratio['total'] = coaches_winratio['won'] + coaches_winratio['lost']
coaches_winratio = coaches_winratio.reset_index()
coaches_winratio.sort_values(by='winratio', ascending=False, inplace=True)

coaches_winratio_sig = coaches_winratio.head(20)
total_games_sum = coaches_winratio_sig['total'].sum()
coaches_winratio_sig['total'] = coaches_winratio_sig['total'] / total_games_sum
plt.figure(figsize=(10, 5))
bar1 = seaborn.barplot(x='coachID', y='winratio', data=coaches_winratio_sig, color='blue')
bar2 = seaborn.barplot(x='coachID', y='total', data=coaches_winratio_sig, color='green')
winratio_patch = Patch(color='blue', label='Win / Loss ratio')
total_patch = Patch(color='green', label='Total games % of top coaches total games')
plt.legend(handles=[winratio_patch, total_patch])
plt.xticks(rotation=90)
plt.title('Top 20 coaches winratio and total games')
plt.xlabel('Coach ID')
plt.ylabel('Win / Loss ratio')
plt.show()

## Playoff appearances

In [ ]:
teams_names = teams['franchID'].unique()

playoffs_number = teams[teams['playoff'] == "Y"]
playoffs_number = playoffs_number.groupby('franchID').size()
playoffs_number = playoffs_number.reset_index(name='playoff_in')

no_playoffs = teams[teams['playoff'] == "N"]
no_playoffs = no_playoffs.groupby('franchID').size()
no_playoffs = no_playoffs.reset_index(name='playoff_out')

playoffs_number = pandas.merge(playoffs_number, no_playoffs, on='franchID', how='outer')

playoffs_number['playoff_in'] = playoffs_number['playoff_in'].fillna(0)
playoffs_number['playoff_out'] = playoffs_number['playoff_out'].fillna(0)

playoffs_number['playoff_in'] = playoffs_number['playoff_in'].astype(int)

playoffs_number['playoff_total'] = playoffs_number['playoff_in'] + playoffs_number['playoff_out']
playoffs_number.sort_values(by='playoff_total', ascending=False, inplace=True)
playoffs_number.sort_values(by='playoff_total', ascending=False, inplace=True)

plt.figure(figsize=(10, 5))
bar1 = seaborn.barplot(x='franchID', y='playoff_total', data=playoffs_number, color='blue')
bar2 = seaborn.barplot(x='franchID', y='playoff_out', data=playoffs_number, color='red')
playoff_patch = Patch(color='blue', label='Playoff appearances')
no_playoff_patch = Patch(color='red', label='Missed playoffs')
plt.legend(handles=[playoff_patch, no_playoff_patch], fontsize=14)
plt.xlabel('Team ID')
plt.ylabel('Number of appearances')
plt.title('Number of playoff appearances and missed playoffs')
plt.show()

## Field goals per rank

In [ ]:
plt.figure(figsize=(10, 5))
seaborn.boxplot(y='o_fgm', x='rank', data=teams, color='blue')
plt.xlabel('Rank')
plt.ylabel('Field goals made')
plt.title('Field goals made vs rank')
plt.show()

## Correlation of features with next year playoff

In [ ]:
correlation = teams_table.drop('franchID', axis=1).corr()
correlation = correlation['next_year_playoff'].reset_index()
correlation.columns = ['feature', 'correlation']
correlation = correlation[(correlation['correlation'] > 0.1) | (correlation['correlation'] < -0.1)]
correlation = correlation.sort_values(by='correlation', ascending=False)
correlation = correlation[correlation['feature'] != 'next_year_playoff']

plt.figure(figsize=(10, 5))
bar = seaborn.barplot(x='feature', y='correlation', data=correlation, color='blue')
plt.xticks(rotation=90)
plt.title('Correlation of features with next year playoffs')
plt.ylabel('Correlation')
plt.xlabel('Feature')
plt.show()


## Trends of features over time

In [ ]:
grouped = teams_table.drop('franchID', axis=1).groupby('year').mean().reset_index()

grouped['awards_won'] = grouped['awards_won'] * 10

plt.figure(figsize=(12, 6))
seaborn.lineplot(data=grouped, x='year', y='awards_won', label='Awards Won 10x')
seaborn.lineplot(data=grouped, x='year', y='average_height', label='Average Height')
seaborn.lineplot(data=grouped, x='year', y='average_weight', label='Average Weight')

# Adding labels and legend
plt.title("Feature Trends Over Time", fontsize=16)
plt.xlabel("Season", fontsize=12)
plt.ylabel("Feature Values", fontsize=12)
plt.legend(title="Features", fontsize=10)
plt.show()

## Distribution of average height and weight for playoff and non-playoff teams

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
seaborn.boxplot(data=teams_table, x='next_year_playoff', y='average_height', palette='Set3')
plt.title('Distribution of Average Height', fontsize=14)
plt.xticks([0, 1], ['Non-Playoff Teams', 'Playoff Teams'])
plt.xlabel('Playoff Status', fontsize=12)
plt.ylabel('Average Height', fontsize=12)
plt.ylim(65, 75)

# Average Weight
plt.subplot(1, 2, 2)
seaborn.boxplot(data=teams_table, x='next_year_playoff', y='average_weight', palette='Set3')
plt.title('Distribution of Average Weight', fontsize=14)
plt.xticks([0, 1], ['Non-Playoff Teams', 'Playoff Teams'])
plt.xlabel('Playoff Status', fontsize=12)
plt.ylabel('Average Weight', fontsize=12)
plt.ylim(140, 190)

plt.tight_layout()
plt.show()